[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/28_moe.ipynb)

# 🔴 Hard: Mixture of Experts (MoE)

Implement a **Mixture of Experts** layer (Mixtral / Switch Transformer style).

### Signature
```python
class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, S, D) -> (B, S, D)
```

### Architecture
- `self.router`: `nn.Linear(d_model, num_experts)` — gating network
- `self.experts`: `nn.ModuleList` of MLPs `(Linear→ReLU→Linear)`
- For each token: select top-k experts, compute weighted sum of their outputs

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.4 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

In [9]:
# ✏️ YOUR IMPLEMENTATION HERE

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()

        self.router=nn.Linear(d_model,num_experts)
        self.num_experts=num_experts
        self.d_model=d_model
        self.top_k=top_k




        self.experts=nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model,d_ff),
                nn.ReLU(),
                nn.Linear(d_ff,d_model)
            ) for _ in range(num_experts)]
        )


    def forward(self, x):
        orig_shape=x.shape
        x=x.view(-1,self.d_model)

        logits=self.router(x)
        weights,indices=torch.topk(logits,self.top_k,dim=-1)

        weights=torch.nn.functional.softmax(weights,dim=-1)

        output=torch.zeros_like(x)

        for i in range(self.top_k):
          exp_indices=indices[:,i]
          exp_weights=weights[:,i].unsqueeze(-1)
          for idx,expert in enumerate(self.experts):
            mask=(exp_indices==idx)
            if mask.any:
              output[mask]+=expert(x[mask])*exp_weights[mask]

        return output.view(*orig_shape)



In [10]:
# 🧪 Debug
moe = MixtureOfExperts(32, 64, num_experts=4, top_k=2)
x = torch.randn(2, 8, 32)
print('Output:', moe(x).shape)
print('Params:', sum(p.numel() for p in moe.parameters()))

Output: torch.Size([2, 8, 32])
Params: 16900


In [11]:
# ✅ SUBMIT
from torch_judge import check
check('moe')


🧪 Testing: Mixture of Experts (MoE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (7.4ms)
  ✅ [2/4] Has router and experts (1.6ms)
  ✅ [3/4] Router logits shape (4.4ms)
  ✅ [4/4] Gradient flow (46.3ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (59.8ms total)
  Progress saved. Run status() to see your dashboard.

